In [1]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from os.path import join as pjoin
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr, zscore, kendalltau
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [2]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/reward_distance'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
pos_distances = np.arange(0, (reward_bin_size*4) + reward_bin_size, reward_bin_size) ## distance from reward locations
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 1 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Get the distance of each cell's peak activity is from the reward locations.

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Find the peak of each place field
pf_peaks = np.max(tuning_curves, axis=1)
## Find the spatial bins where each peak occurred
field_dist = np.zeros(pf_peaks.shape[0])
for idx, peak in enumerate(pf_peaks):
    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

pos_vals = []
for start in pos_distances:
    rw_one_amount = np.sum((abs(field_dist - first_rw_pos) >= start) & (abs(field_dist - first_rw_pos) < start + reward_bin_size))
    rw_two_amount = np.sum((abs(field_dist - second_rw_pos) >= start) & (abs(field_dist - second_rw_pos) < start + reward_bin_size))
    pos_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

all_vals = []
for pos in all_midpoints:
    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))
    all_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

In [ ]:
## Example proportion of place fields for all distances centered at x-axis values
fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
fig.add_trace(go.Scattergl(x=all_midpoints * conversion, y=all_vals, mode='lines+markers', marker_size=8))
fig.update_yaxes(range=[0, np.max(all_vals) + 0.01])
fig.show()

In [ ]:
## Example proportion of place fields for only positive differences
fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
fig.add_trace(go.Scattergl(x=pos_distances * conversion, y=pos_vals))
fig.update_yaxes(range=[0, 0.16])
fig.show()

In [ ]:
## Example distribution of place fields across track
rw_bins = np.arange(0, 6.28 + reward_bin_size, reward_bin_size)
H, xbin = np.histogram(field_dist, bins=rw_bins)
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Proportion Place Fields', width=600)
hnorm = H / np.sum(H) ## convert to proportion
fig.add_trace(go.Bar(x=xbin, y=hnorm, marker_color=ce_colors_dict['Multi-context'], marker_line_width=2, marker_line_color='black'))
for val in [reward_one_pos, reward_two_pos]:
    fig.add_vline(x=val, line_dash='dash', line_width=3, opacity=0.7, line_color='red')
fig.update_yaxes(range=[0, 0.05])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_distribution_of_place_fields_running{only_running}_cordir{correct_dir}.png'), width=500, height=500)

### Combine across mice.

In [3]:
## Settings
only_running = True
correct_dir = True
cell_type = 'place_cells'

In [4]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion': [],
               'lick_accuracy': [], 'rewards': [], 'hr': [], 'cr': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        behav_path = pjoin(dpath, f'{experiment}/output/behav/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            behav_mouse_path = pjoin(behav_path, f'{mouse}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Behavior
                behav = pd.read_feather(pjoin(behav_mouse_path, f'{mouse}_{session.split('_')[-1].split('.')[0]}.feat'))
                behav = behav[~behav['probe']] ## exclude probe
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]    
                lick_acc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                signal = pd.DataFrame(ctb.dprime_metrics(behav, mouse, day=index+1, reward_ports=[reward_one, reward_two], forward_reverse='all'))
                
                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
                    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['proportion'].append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])
                    output_dict['lick_accuracy'].append(lick_acc)
                    output_dict['rewards'].append(np.sum(behav['water']))
                    output_dict['hr'].append(signal.groupby(['day'], as_index=False).agg({'hits': 'mean'})['hits'].values[0])
                    output_dict['cr'].append(signal.groupby(['day'], as_index=False).agg({'CR': 'mean'})['CR'].values[0])
rel_df = pd.DataFrame(output_dict)
rel_df.to_feather(pjoin(int_data, f'max_pf_relative_rewards_{cell_type}_second_def.feat'))

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
## Plot proportion of place fields for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['proportion']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['proportion']['sem'])))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_proportion_pfs.png'), width=500, height=500)

In [15]:
## Look at development of place field representation in A for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
cont = avg[(avg['day'] >= 1) & (avg['day'] < 6)] ## subset for days just in context A
for day in cont['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = cont[(cont['day'] == day) & (cont['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], showlegend=False, mode='lines+markers',
                            legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2, marker_size=7,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'])), row=1, col=day)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.09])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
# fig.update_xaxes(range=[-sub_rel_bins - 2, sub_rel_bins + 2])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=-0.5,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (cm)':
        fig['layout']['annotations'][idx]['yshift'] = -80
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_contA_multi_two_over_rep.png'), width=1400, height=500)

In [16]:
## Plot the first day in each new context for Multi-context mice
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields', titles=['Multi-context'])
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    d_data = avg[(avg['day'] == day) & (avg['group'] == 'Multi-context')]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], mode='lines+markers',
                            name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=7,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'])))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_context_switches_mc.png'), width=500, height=500)

In [17]:
## Plot the first day in each new context for Two-context and Multi-context
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='', titles=['Two-context', 'Multi-context'], rows=1, columns=2,
                               shared_x=True, shared_y=True, width=900)
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    for idx, group in enumerate(['Two-context', 'Multi-context']):
        d_data = avg[(avg['day'] == day) & (avg['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], mode='lines+markers', showlegend=False,
                                legendgroup=f'Day {day}', name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=7,
                                marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'])), row=1, col=idx + 1)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.data[4]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_context_switches_tc_mc.png'), width=1000, height=500)

In [22]:
## Correlate rewards with the proportion of place fiels around the reward location
day = 6
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Rewards', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['rewards'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['rewards'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=25)
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_rewards_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['rewards'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['rewards'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

Two-context: PearsonRResult(statistic=np.float64(0.4693287511592526), pvalue=np.float64(0.2880126903575772))
Multi-context: PearsonRResult(statistic=np.float64(0.7098641384261282), pvalue=np.float64(0.04854281866940099))


In [26]:
## Correlate lick accuracy with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Lick Accuracy (%)', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['lick_accuracy'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['lick_accuracy'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=25, range=[0, 100])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_lick_accuracy_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['lick_accuracy'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['lick_accuracy'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

Two-context: PearsonRResult(statistic=np.float64(0.6280515605120125), pvalue=np.float64(0.13096290805789398))
Multi-context: PearsonRResult(statistic=np.float64(0.5433627770599035), pvalue=np.float64(0.16396339369401486))


In [30]:
## Correlate hit rate with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Hit Rate', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['hr'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['hr'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=0.25, range=[0, 1])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_hr_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['hr'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['hr'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

Two-context: PearsonRResult(statistic=np.float64(0.7954774241607643), pvalue=np.float64(0.03243287456867557))
Multi-context: PearsonRResult(statistic=np.float64(0.47588578431919126), pvalue=np.float64(0.233277243838073))


In [34]:
## Correlate correct rejection rate with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Correct Rejection Rate', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cr'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['cr'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=0.25, range=[0, 1])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_cr_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['cr'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['cr'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

Two-context: PearsonRResult(statistic=np.float64(0.47943589724920793), pvalue=np.float64(0.2763071238693641))
Multi-context: PearsonRResult(statistic=np.float64(0.3755018032767106), pvalue=np.float64(0.3593174942275533))


### Get the proportion of place fields around undershooting and overshooting locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
port_type = 'overshooting'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)
                    
                ## Get positions of the under or over-shooting ports and use those as the zero location
                front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
                if port_type == 'overshooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
                elif port_type == 'undershooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)
                
                ## Find distance from port locations
                rw_one_dist = abs(field_dist - first_rw_pos)
                rw_two_dist = abs(field_dist - second_rw_pos)

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
                    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['proportion'].append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])
rel_df = pd.DataFrame(output_dict)
# rel_df.to_feather(pjoin(int_data, f'max_pf_relative_rewards_{cell_type}_second_def.feat'))

In [ ]:
## Plot proportion of place fields for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields', width=500)
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['proportion']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['proportion']['sem'])))
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_{port_type}_mc_tc_proportion_pfs.png'), width=500, height=500)

In [ ]:
## Look at development of place field representation in A for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
cont = avg[(avg['day'] >= 1) & (avg['day'] < 6)] ## subset for days just in context A
for day in cont['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = cont[(cont['day'] == day) & (cont['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], showlegend=False, mode='lines+markers',
                            legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2, marker_size=7,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'])), row=1, col=day)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
# fig.update_xaxes(range=[-sub_rel_bins - 2, sub_rel_bins + 2])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=-0.5,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (cm)':
        fig['layout']['annotations'][idx]['yshift'] = -80
fig.show()
fig.write_image(pjoin(fig_path, f'contA_multi_two_over_rep_{port_type}.png'), width=1400, height=500)

### Get the average amount of spatial information of cells x distance from the rewarded locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 
               'avg_spatial_info_rw1': [], 'avg_spatial_info_rw2': [], 'avg_spatial_info': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['avg_spatial_info_rw1'].append(np.mean(sdata['skaggs_info'][rw1_bool].values))
                    output_dict['avg_spatial_info_rw2'].append(np.mean(sdata['skaggs_info'][rw2_bool].values))
                    output_dict['avg_spatial_info'].append(np.mean(np.concatenate((sdata['skaggs_info'][rw1_bool].values, sdata['skaggs_info'][rw2_bool].values))))
si_df = pd.DataFrame(output_dict)
si_df.to_feather(pjoin(int_data, f'spatial_info_relative_rewards_{cell_type}_second_def.feat'))

In [ ]:
## Plot the average spatial information for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = si_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Spatial Information (bits/event)')
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['avg_spatial_info']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['avg_spatial_info']['sem'])))
fig.update_yaxes(range=[2.6, 6.6])
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_avg_spatial_info_pfs.png'), width=500, height=500)

### Get spatial information around under/overshooting reward locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)
                    
                ## Get positions of the under or over-shooting ports and use those as the zero location
                front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
                if port_type == 'overshooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
                elif port_type == 'undershooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)
                
                ## Find distance from port locations
                rw_one_dist = abs(field_dist - first_rw_pos)
                rw_two_dist = abs(field_dist - second_rw_pos)

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
                    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['proportion'].append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])
rel_df = pd.DataFrame(output_dict)
# rel_df.to_feather(pjoin(int_data, f'max_pf_relative_rewards_{cell_type}_second_def.feat'))

### Find average trial start of place fields across reward distances.

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'avg_trial_start': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)

                    sub_ar = sdata[(rw1_bool) | (rw2_bool)]
                    trial_raster, raster_bins = pc.trial_raster(sub_ar, bin_size=bin_size, correct_dir=correct_dir, only_running=only_running)
                    trials_start_per_neuron = pc.place_field_starting_trials(trial_raster)

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['avg_trial_start'].append(np.round(np.mean(trials_start_per_neuron)))
trial_start_df = pd.DataFrame(output_dict)
trial_start_df.to_feather(pjoin(int_data, f'trial_start_relative_rewards_{cell_type}_second_def.feat'))

In [ ]:
## Plot the average stable place field trial start for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = trial_start_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_trial_start': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Place Field Trial Start')
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['avg_trial_start']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['avg_trial_start']['sem'])))
fig.update_yaxes(range=[0, 20]) ## y-axes for day 16
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_avg_trial_start_pfs.png'), width=500, height=500)